# 01 — RAG Foundations: Space Facts Explorer

The **"under the hood"** notebook — no vector database, no LLM, no API key needed.

We build every piece of RAG from scratch so you can see exactly what is happening:

| Step | What we do |
|------|-----------|
| 1 | Create `Document` objects from plain text |
| 2 | Split them into smaller chunks |
| 3 | Convert chunks into numeric vectors (embeddings) |
| 4 | Search by computing cosine similarity by hand |

### What is RAG?
```
User Question
     │
     ▼
[Embed question] ──cosine similarity──▶ [Chunk database]
                                              │
                                         top-k chunks (context)
                                              │
                                         [LLM] ──▶ Answer
```
RAG grounds LLM responses in real documents, reducing hallucinations.

## 1. Install Dependencies

In [ ]:
# Run once if not installed
# !pip install langchain langchain-community sentence-transformers numpy

## 2. Imports

In [ ]:
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import numpy as np

print("All imports OK!")

## 3. Create Documents

A `Document` has two parts:
- `page_content` — the actual text
- `metadata` — extra info like source, topic, page number

In [ ]:
# We use a subset of space_facts.txt broken into sections
raw_texts = [
    ("The Sun is a G-type main-sequence star at the centre of our solar system. "
     "It accounts for 99.86% of the total mass of the solar system. "
     "Its surface temperature is about 5,500°C, while the core reaches 15 million°C. "
     "Light from the Sun takes 8 minutes 20 seconds to reach Earth."),

    ("Mercury is the smallest planet and closest to the Sun, with temperatures "
     "swinging from -180°C at night to 430°C during the day. Venus is the hottest "
     "planet with surface temperatures around 465°C due to its thick CO2 atmosphere. "
     "Mars has the largest volcano in the solar system, Olympus Mons."),

    ("Jupiter is the largest planet — its mass exceeds all other planets combined. "
     "Its Great Red Spot is a storm raging for over 300 years. Saturn is famous for "
     "its ring system made of ice and rock. Neptune has the fastest winds in the "
     "solar system, reaching 2,100 km/h."),

    ("A black hole is a region where gravity is so strong that nothing, including "
     "light, can escape past the event horizon. Black holes form when massive stars "
     "collapse. The Milky Way's central black hole, Sagittarius A*, has a mass "
     "4 million times that of the Sun."),

    ("The Space Age began on 4 October 1957 with the Soviet Union's Sputnik 1. "
     "On 20 July 1969, Neil Armstrong and Buzz Aldrin became the first humans to "
     "walk on the Moon during Apollo 11. The Voyager 1 spacecraft, launched 1977, "
     "entered interstellar space in 2012."),

    ("Stars are born inside giant clouds of gas and dust called nebulae. The Milky Way "
     "contains between 200-400 billion stars. The nearest galaxy is Andromeda, 2.5 "
     "million light-years away. The observable universe contains roughly two trillion "
     "galaxies. The universe began 13.8 billion years ago with the Big Bang."),
]

# Wrap each text in a Document with metadata
docs = [
    Document(page_content=text, metadata={"source": "space_facts.txt", "section": i})
    for i, text in enumerate(raw_texts)
]

print(f"Created {len(docs)} documents")
print(f"\nExample document:")
print(f"  Content : {docs[0].page_content[:80]}...")
print(f"  Metadata: {docs[0].metadata}")

## 4. Text Splitting

Documents are split into smaller **chunks** because:
- LLMs have a limited context window
- Smaller chunks = more precise retrieval

`RecursiveCharacterTextSplitter` tries to split at paragraph → sentence → word level.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,       # max characters per chunk
    chunk_overlap=40,     # overlap to avoid cutting off context
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(docs)

print(f"Split {len(docs)} documents → {len(chunks)} chunks")
print(f"\nChunk 0:")
print(f"  Content : {chunks[0].page_content}")
print(f"  Metadata: {chunks[0].metadata}")

## 5. Embeddings

An **embedding** converts text into a vector of numbers so we can do maths on meaning.
Similar sentences have vectors that point in similar directions.

In [ ]:
# Load a small but capable model (downloads ~90 MB the first time)
print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")

# Embed all chunks
chunk_texts = [c.page_content for c in chunks]
embeddings  = model.encode(chunk_texts, show_progress_bar=True)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Each chunk → {embeddings.shape[1]}-dimensional vector")
print(f"\nFirst 8 values of chunk 0 embedding: {embeddings[0][:8].round(4)}")

## 6. Cosine Similarity Search

**Cosine similarity** measures how closely two vectors point in the same direction.
Score = 1.0 means identical meaning, 0.0 means unrelated.

```
similarity = (A · B) / (|A| × |B|)
```

In [ ]:
def cosine_search(query: str, embeddings: np.ndarray,
                  texts: list, top_k: int = 3) -> list:
    """Embed a query and find the most similar chunks."""
    q_emb = model.encode([query])                          # shape (1, 384)
    # dot product for each chunk
    dots  = np.dot(embeddings, q_emb.T).squeeze()         # shape (n_chunks,)
    norms = np.linalg.norm(embeddings, axis=1) * np.linalg.norm(q_emb)
    scores = dots / (norms + 1e-8)                        # cosine similarity

    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(texts[i], float(scores[i])) for i in top_idx]

## 7. Try It Out!

In [ ]:
queries = [
    "How hot is the Sun?",
    "Which planet has the biggest storm?",
    "What is a black hole?",
    "When did humans first walk on the Moon?",
]

for query in queries:
    print(f"\n{'='*55}")
    print(f"  Query: {query}")
    print(f"{'='*55}")
    results = cosine_search(query, embeddings, chunk_texts, top_k=2)
    for rank, (text, score) in enumerate(results, 1):
        print(f"  [{rank}] Score: {score:.3f} | {text[:120]}...")

## 8. Key Takeaways

| Concept | What you learned |
|---------|-----------------|
| `Document` | Container for text + metadata |
| `TextSplitter` | Breaks documents into retrievable chunks |
| `SentenceTransformer` | Converts text to numeric vectors |
| Cosine similarity | Measures semantic closeness between vectors |
| Retrieval | Finding the most relevant chunks for a query |

**Next notebook →** replace the manual numpy search with a real vector database (FAISS).